# MLB Schedule & Game Data

This notebook demonstrates the MLB Stats API integration in `polars-baseball`:
schedules, rosters, standings, and box scores — all returning native Polars DataFrames.

In [ ]:
from __future__ import annotations

import polars as pl

import polars_baseball as pb

## 1. Daily schedule

Fetch all games for a given date.

In [ ]:
schedule = await pb.mlb.schedule(date="2026-06-01")
print(f"Games on 2026-06-01: {schedule.height}")
schedule.select(["game_date", "teams_home_team_name", "teams_away_team_name", "venue_name"]).head(10)

## 2. Division standings

Standings return one row per team in each division with W/L, GB, and streak.

In [ ]:
standings = await pb.standings(season=2026, league_id="103")
standings.select(
    [
        "division_name",
        "team_name",
        "wins",
        "losses",
        "games_back",
        "streak",
    ]
).sort("division_name", "games_back")

## 3. Team roster

Fetch the active roster for a specific team by `team_id`.  
`team_ids()` returns a mapping of team abbreviations to IDs.

In [ ]:
teams = await pb.team_ids()
print(teams.filter(pl.col("abbreviation") == "LAD"))

In [ ]:
roster = await pb.mlb.roster(team_id=119, season=2026)
roster.select(["full_name", "primary_position_abbreviation", "bat_side_code", "pitch_hand_code"]).head(15)

## 4. Team metadata lookup

Fetch all teams, venues, divisions, and leagues as dimension tables — useful for joins.

In [ ]:
teams_df = await pb.mlb.teams()
teams_df.select(["id", "name", "abbreviation", "league_name", "division_name"]).head(5)

## Summary

- MLB Stats API endpoints return structured `polars.DataFrame` results.
- Schedule, standings, rosters, and metadata are all accessible from `pb.mlb.*`.
- Team IDs can be resolved dynamically via `pb.team_ids()`.

Next steps:
- Try `pb.mlb.game_boxscore(game_pk=...)` for per-game detail
- Try `pb.mlb.game_play_by_play(game_pk=...)` for play-by-play logs
- See [notebooks/statcast_pitch_mix_demo.ipynb](statcast_pitch_mix_demo.ipynb)